# CBF-U Model

This notebook contains a step-by-step implementation of the CBF-U model, along with further execution to explore the bias. This notebook is a part of the "Job Recommendation and the Role of Stereotypes" paper by Chervinskyy and Meliushko, precisely, it's the CBF-U model.

![alt text](images/cbfu.png)

### CBF-U Architecture:

Content-based Filtering that takes into account the user's limited personal characteristics. (CBF-U for short). The CBF-U pipeline is described in the "proposed models" section and is further illustrated in Figure above. The user prompt is constructed and then fed into an *JinaAI* embedding model, which is an *Sentence-BERT* (*SBERT*)-based model. The low-dimensional representation of a user is put through n Feedforward Neural Network (*FNN*) layers. Finally, using cosine similarity, we determine which job vacancy best suits the user. The job is processed through the same pipeline of sentence construction, utilizing existing features such as a description, title, salary, and required experience level. Then all the sentences created are fed to an LLM (*JinaAI* embedding model) to produce a low-dimensional vector.

### Abstract
Despite the active use of Job recommender systems, the reinforcement of social stereotypes remains an under-investigated issue. Prior work shows that gendered wording in job ads and biased Natural Language Processing (NLP) based algorithms can influence which candidates or jobs are recommended. The model introduces bias through both explicit values and those derived from the data, without any explicit specifications. In this research, we constructed an NLP-based job recommender system utilizing various datasets to investigate how different attributes of applicants, including specific writing stylistic parameters, influence the jobs for which they are recommended. Our goal is to identify where bias enters the recommendation process and suggest ways to mitigate it, which would also help us identify core features of each applicant responsible for their recommendation outcome. The results show that a significant amount of bias is introduced by simply adding noise as a biased sentence to the original prompt about the user. This signifies the potential liability of Sentence-BERT (SBERT)-style sentence embedding models, specifically the *JinaAI* embedding model used in this study, when exposed to biased or demographically salient input data.

### Imports

The CBF-U model was implemented in Python using *transformers* and *PyTorch*. All the additional imports are in the python files attached, that are imported to this notebook.

In [1]:
import torch
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, BertTokenizer, BertForSequenceClassification
from torch.utils.data import DataLoader, TensorDataset

Ignore warnings

In [2]:
import warnings
warnings.filterwarnings("ignore")

## Data download, preprocessing and embedding

In this research, we utilize 2 following datasets:

 - **CareerBuilder**. It includes users, job postings, and observed applications, organized into seven time windows with defined training and test periods. This structure provides ground-truth behavioral labels enabling the standard offline evaluation of ranking performance. The user table contains demographic and professional attributes. Possibly biased attributes, such as gender, age, and political beliefs, are lacking. While this is a positive aspect of the data set, we do not have true values for the bias investigation. 

 - **LinkedIn Job Postings (2023 - 2024)**. It is a dataset with high-quality semantic descriptions of jobs containing textual fields (job title, description, requirements, and related metadata). This dataset is suitable for embedding-based retrieval as it provides natural-language content that can be transformed into dense embeddings for improved semantic matching. It supports analysis of language-driven bias, where gender-coded or age-coded cues in descriptions might influence which jobs appear most similar to a given user profile. Due to the high-quality semantic textual description of the jobs, it enables a deeper analysis of metastereotype-sensitive content, providing more insight into pre-trained models' analysis of the text. 

In [3]:
from dataset import load_data

### CareerBuilder - jobs

Load `apps.tsv` (interactions), `users.tsv`, `jobs.tsv`, and `user_history.tsv` (history of previous jobs)

In [4]:
interactios =   load_data('apps.tsv')
users =         load_data('users.tsv')
jobs =          load_data('jobs.tsv')
user_history =  load_data('user_history.tsv')

We now prepare the data for reprocessing by removing the HTML parts of the original text and constructing the prompt described in the research paper. The training data-items for the **CareerBuilder** dataset, the prompt looks as follows:

"Title: {title}. Description: {description}. Requirements: {requirements}."

This is done in the function `compute_all_st_jobs`, as well as further data embedding. For this, we use *jinaai/jina-embeddings-v2-small-en*, as we aim to preserve all information about the user. After conducting a statistical analysis of the data, we concluded that average ST models, such as *sentence-transformers/all-MiniLM-L6-v2*, would discard the data, as the token limit is too short. Therefore, we used the JinaAI model instead.

In [5]:
model_name = "jinaai/jina-embeddings-v2-small-en"
tokenizer = AutoTokenizer.from_pretrained(model_name,trust_remote_code=True)
model = AutoModel.from_pretrained(model_name,trust_remote_code=True)

device = "cuda"  
model.to(device)  
model.eval()

JinaBertModel(
  (embeddings): JinaBertEmbeddings(
    (word_embeddings): Embedding(30528, 512, padding_idx=0)
    (token_type_embeddings): Embedding(2, 512)
    (LayerNorm): LayerNorm((512,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): JinaBertEncoder(
    (layer): ModuleList(
      (0-3): 4 x JinaBertLayer(
        (attention): JinaBertAttention(
          (self): JinaBertSelfAttention(
            (query): Linear(in_features=512, out_features=512, bias=True)
            (key): Linear(in_features=512, out_features=512, bias=True)
            (value): Linear(in_features=512, out_features=512, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (output): JinaBertSelfOutput(
            (dense): Linear(in_features=512, out_features=512, bias=True)
            (LayerNorm): LayerNorm((512,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
 

Due to limited resources, we work only with the 2nd window, as the preprocessing tasks take approximately 8 hours on the hardware available for this research.

In [6]:
from models import compute_all_st_jobs

In [ ]:
jobs_w2 = jobs.loc[jobs['WindowID'] == 2]
semantics = compute_all_st_jobs(jobs_w2, model, tokenizer, device, 16, 2048, "window2")

155161it [00:57, 2679.96it/s]
100%|██████████| 9698/9698 [7:40:43<00:00,  2.85s/it]   


In [153]:
semantics =     np.genfromtxt('data/window2.csv', delimiter=',')

### CareerBuilder - users

We use `users.tsv` and `user_history.tsv` loaded above.

In [8]:
users_w2 = users.loc[users['WindowID'] == 2]
user_history_w2 = user_history.loc[user_history['WindowID'] == 2]

Now, for clarification, consider the random user and investigate them as an example of prompt generation. 

In [9]:
from dataset import user_create_prompt
from models import compute_all_st_users

In [10]:
user_create_prompt(users_w2.iloc[45376],user_history_w2)

"I have a Associate's education, where I got a major in Business Management in 2012. I have 7 years of experience and have held 3 roles. Work history includes: Customer service and Acquisitions supervisor; Customer Service Representative Supervisor; Administrative Assistant. I have people-management experience leading a team of 10 people."

In [11]:
users_semantics = compute_all_st_users(users_w2, user_history, model, tokenizer, batch_size=64,max_length=1024, name="window2")

58228it [02:00, 482.91it/s]
100%|██████████| 910/910 [01:40<00:00,  9.03it/s]


### CareerBuilder - interactions

Now that we have preprocessed the required data, we construct a simple prediction model to further investigate it. We achieve this via the `get_jobs_with_interactions` function. We delete any users not present in the interactions, all jobs that are missing in the interactions, and we further derive an interaction list that can be used as a training dataset, clean of any missing values.

In [12]:
from dataset import UBERTDataset, get_jobs_with_interactions

In [163]:
interactios_w2 =    interactios[interactios["WindowID"] == 2].reset_index(drop=True)
jobs_w2 =           jobs[jobs["WindowID"] == 2].reset_index(drop=True)
users_w2 =          users[users['WindowID'] == 2].reset_index(drop=True)

In [168]:
final_jobs_df, interactions_clean,missing_jobids, users_clean, missing_userids, final_users_df  = get_jobs_with_interactions(semantics, jobs_w2, interactios_w2, users_w2, users_semantics)
dataset = UBERTDataset(final_jobs_df, interactions_clean, final_users_df)

### CareerBuilder - Dataset

get Interactions:

In [169]:
train_interactions = interactions_clean[interactions_clean["Split"] == "Train"]
test_interactions  = interactions_clean[interactions_clean["Split"] == "Test"]

get Dataset:

In [170]:
train_dataset = UBERTDataset(jobs=final_jobs_df, interactions=train_interactions, users=final_users_df)
test_dataset = UBERTDataset(jobs=final_jobs_df, interactions=test_interactions, users=final_users_df)

get Dataloader:

In [171]:
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
    drop_last=True
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False
)

## CBF-U Model - implementation and training

For this study, we create a CBF-U supervised UserAdapter with a dimensionality of 512. The Architecture is described in CBF-U Architecture (at the start of this notebook).

In [172]:
from models import UserAdapter
from train import train_model

In [173]:
model_new = UserAdapter(dim=512).to("cuda")

train the model:

In [174]:
user_model = train_model(
    model_new,
    train_loader,
    test_loader,
    num_epochs=30,
    earl_stop=2,
    temperature=0.1,
    lr=0.001
)

  3%|▎         | 1/30 [01:23<40:17, 83.37s/it]

Epoch 1: Train Loss = 5.5252, Val Loss = 5.5369


  7%|▋         | 2/30 [02:47<39:08, 83.87s/it]

Epoch 2: Train Loss = 5.5117, Val Loss = 5.5374


  7%|▋         | 2/30 [04:11<58:46, 125.93s/it]

Epoch 3: Train Loss = 5.5050, Val Loss = 5.5410
Early stopping


## Bias Investigation - correlations with other explicit attributes

For the evaluation, as our task is not to improve the model indefinitely, but to research bias, we use correlations with other explicit attributes. Our primary focus is to identify the effect of a specific noise signal on the prediction; therefore, we aim here to construct two tables: a bias audit table, comprising various bias introductions and the corresponding effects of this data included in the candidate information. This table will examine the correlation between the noise signal introduced (bias signal) and salary. The second table constructed should display the correlation between the bias signal and readability and personality attributes, derived from the text.

In [175]:
from models import compute_all_st_jobs_linkedin

We exclude all items with *NaN* values in the description, title, normalized salary, company name, and formatted experience level to ensure that only records with complete information are used for comparison. As the description length varies substantially across the dataset, we further exclude all items with fewer than 250 words or more than 750 words in the description. To provide data about the description statistics: the median length is 280 words (rounded to the nearest ten), the upper quartile is 470 words, and the 95th percentile is 910 words.

### Load data - LinkedIn Job Postings (2023 - 2024)

In [176]:
postings_linkedin = pd.read_csv('data/postings.csv')

In [177]:
cols_required = ["description", "title", "normalized_salary", "company_name", "formatted_experience_level"]
postings_linkedin_edited = postings_linkedin.dropna(subset=cols_required)
postings_linkedin_edited = postings_linkedin_edited[postings_linkedin_edited["description"].str.split().str.len().between(250, 750)]
postings_linkedin_edited = postings_linkedin_edited[postings_linkedin_edited["skills_desc"].isna()]
postings_linkedin_edited = postings_linkedin_edited[postings_linkedin_edited["work_type"] == "FULL_TIME"]
postings_linkedin_edited = postings_linkedin_edited[postings_linkedin_edited["pay_period"] == "YEARLY"]
postings_linkedin_edited = postings_linkedin_edited[postings_linkedin_edited["currency"] == "USD"]
postings_linkedin_edited = postings_linkedin_edited.dropna(subset=["company_name"])

In [ ]:
linkedin_semantics = compute_all_st_jobs_linkedin(postings_linkedin_edited, model, tokenizer, "cuda", 18, 2048, "linkedin")

100%|██████████| 428/428 [18:34<00:00,  2.60s/it]


### Define Bias to explore.

In [179]:
specs = [
        ("Religious identification", ["religious", "non-religious"]),
        ("Age group", ["18-28", "29-42", "43-64"]),
        ("Gender", ["male", "female", "non-binary"]),
        ("Political alignment", ["liberal", "moderate", "conservative"]),
        ("Immigration status", ["citizen", "non-citizen"]),
        ("Language proficiency:", ["native speaker", "non-native speaker"]),
        ("Disability status",["has a disability", "no disability"])
    ]

## Table 1 - Bias Audit

This presents a comparison of the average salary and the mode of experience level with respect to the randomly introduced biased sentence.

We aim to answer a question:

 - **How biased is the model with respect to different introduced bias attributes such as gender?**

In [ ]:
from bias_analysis import run_bias_suite
from bias_analysis import results_to_table

An extensive bias suite includes 5,000 users and k = 50 items to recommend.

In [188]:
postings_linkedin_edited.rename(columns={"job_id": "JobID"}, inplace=True)

In [207]:
results_bigger = run_bias_suite(user_model, model, tokenizer, users, linkedin_semantics, postings_linkedin_edited, specs = specs, n_users=5000, k=50, seed=42)

100%|██████████| 7/7 [05:54<00:00, 50.68s/it]


In [285]:
table = results_to_table(results_bigger, postings_linkedin_edited)

In [286]:
final_table = pd.concat([table[table["Group"]=="none"], table[table["Group"]!="none"].sort_values(["Bias Category","Group"])], ignore_index=True)
final_table

,Bias Category,Group,Avg. Salary,Mode Experience
0,Religious identification,none,"116,052",Mid-Senior level
1,Age group,none,"116,052",Mid-Senior level
2,Gender,none,"116,052",Mid-Senior level
3,Political alignment,none,"116,052",Mid-Senior level
4,Immigration status,none,"116,052",Mid-Senior level
5,Language proficiency:,none,"116,052",Mid-Senior level
6,Disability status,none,"116,052",Mid-Senior level
7,Age group,18-28,"116,191",Mid-Senior level
8,Age group,29-42,"119,287",Mid-Senior level
9,Age group,43-64,"115,759",Mid-Senior level


### Bias Audit Results

Average normalized salary (rounded to thousands) and most frequent experience level among top-K recommended jobs.

- K = 50  
- Users per group = 5000  

---

#### Baseline (no demographic attribute)

| Bias Category | Group | Avg. Salary | Mode Experience |
|---------------|-------|-------------|-----------------|
| None | none | 116,000 | Mid-Senior level |


#### Religious identification

| Bias Category | Group | Avg. Salary | Mode Experience |
|---------------|-------|-------------|-----------------|
| Religious identification | non-religious | 111,000 | Mid-Senior level |
| Religious identification | religious | 111,000 | Mid-Senior level |


#### Age group

| Bias Category | Group | Avg. Salary | Mode Experience |
|---------------|-------|-------------|-----------------|
| Age group | 18–28 | 116,000 | Mid-Senior level |
| Age group | 29–42 | 119,000 | Mid-Senior level |
| Age group | 43–64 | 116,000 | Mid-Senior level |


#### Gender

| Bias Category | Group | Avg. Salary | Mode Experience |
|---------------|-------|-------------|-----------------|
| Gender | female | 115,000 | Mid-Senior level |
| Gender | male | 121,000 | Mid-Senior level |
| Gender | non-binary | 122,000 | Mid-Senior level |


#### Political alignment

| Bias Category | Group | Avg. Salary | Mode Experience |
|---------------|-------|-------------|-----------------|
| Political alignment | conservative | 132,000 | Mid-Senior level |
| Political alignment | liberal | 133,000 | Mid-Senior level |
| Political alignment | moderate | 127,000 | Mid-Senior level |


#### Language proficiency

| Bias Category | Group | Avg. Salary | Mode Experience |
|---------------|-------|-------------|-----------------|
| Language proficiency | native speaker | 110,000 | Mid-Senior level |
| Language proficiency | non-native speaker | 111,000 | Mid-Senior level |


#### Disability status

| Bias Category | Group | Avg. Salary | Mode Experience |
|---------------|-------|-------------|-----------------|
| Disability status | has a disability | 110,000 | Mid-Senior level |
| Disability status | no disability | 110,000 | Mid-Senior level |

Tables above present a comparison of the average salary and the mode of experience level with respect to the randomly introduced biased sentence. The different groups exhibit a relatively large number of differences. We observe that simply adding a sentence specifying a particular attribute to the model, regardless of the attribute's value, already affects the results. For example, when a "Political alignment" statement is present in the profile, irrespective of its value, the average recommended salary increases noticeably. Mode Experience is not affected by the bias.

## Table - 2 (personality and readability with bias noise)

We do that with a smaller result set, which is less demanding on computational power. 

We aim to answer the following question:

 - **How is bias distributed with respect to abstract text-features, such as readability and personality?**

In [287]:
from bias_analysis import summarize_bias_personality_readability, personality_detection

In [289]:
results_smaller = run_bias_suite(user_model,model, tokenizer, users, linkedin_semantics, postings_linkedin_edited, specs = specs, n_users=50, k=20, seed=42)

  0%|          | 0/7 [00:00<?, ?it/s]

100%|██████████| 7/7 [00:03<00:00,  1.83it/s]


In [290]:
_personality_tokenizer = BertTokenizer.from_pretrained("Minej/bert-base-personality")
_personality_model = BertForSequenceClassification.from_pretrained("Minej/bert-base-personality")
_personality_model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [293]:
summary = summarize_bias_personality_readability(
    results=results_smaller,
    jobs_meta=postings_linkedin_edited,
    personality_fn=personality_detection,
    personality_model=_personality_model,
    personailty_tokenizer=_personality_tokenizer,
    k=20,
)

150it [05:21,  2.15s/it]:00<?, ?it/s]
200it [08:04,  2.42s/it]:21<32:11, 322.00s/it]
200it [07:41,  2.31s/it]:26<34:48, 417.68s/it]
200it [07:31,  2.26s/it]:08<29:11, 437.88s/it]
150it [04:23,  1.76s/it]:39<22:09, 443.14s/it]
150it [03:47,  1.52s/it]:03<12:37, 378.53s/it]
150it [03:57,  1.58s/it]:51<05:27, 327.33s/it]
100%|██████████| 7/7 [40:48<00:00, 349.83s/it]


In [294]:
summary

,bias,value,readability,Extroversion,Neuroticism,Agreeableness,Conscientiousness,Openness
0,Age group,18-28,12.044303,0.112060,0.030624,0.216767,0.118188,0.308224
1,Age group,29-42,11.664565,0.109784,0.028579,0.216701,0.125869,0.303751
2,Age group,43-64,10.910111,0.106713,0.031336,0.223916,0.129340,0.303463
3,Disability status,has a disability,8.489650,0.121011,0.042556,0.210398,0.116347,0.323048
4,Disability status,no disability,8.772835,0.115726,0.043555,0.215272,0.125595,0.320420
5,Gender,female,10.346117,0.111895,0.025606,0.225203,0.157273,0.300895
6,Gender,male,11.378771,0.124725,0.028125,0.210198,0.114280,0.314127
7,Gender,non-binary,11.591470,0.115848,0.032270,0.219948,0.143457,0.310932
8,Immigration status,citizen,12.066433,0.096245,0.030480,0.244149,0.157820,0.290716
9,Immigration status,non-citizen,11.033268,0.088844,0.033514,0.257584,0.193778,0.290660
